# **1. Perkenalan Dataset**

Dataset yang digunakan adalah **Social Media & Mental Health** (Kaggle), berisi 8.000 rekaman sintetis yang merepresentasikan perilaku pengguna media sosial serta dampaknya terhadap kesehatan mental.

**Fitur utama:**
- `User_ID`, `Age`, `Gender`, `User_Archetype`, `Primary_Platform`
- `Daily_Screen_Time_Hours`, `Dominant_Content_Type`, `Activity_Type`
- `Late_Night_Usage`, `Social_Comparison_Trigger`, `Sleep_Duration_Hours`
- `GAD_7_Score` (Anxiety 0–21), `GAD_7_Severity`
- `PHQ_9_Score` (Depression 0–27), `PHQ_9_Severity` ← **Target prediksi**

**Tujuan:** Memprediksi `PHQ_9_Severity` (tingkat depresi) berdasarkan pola penggunaan media sosial dan faktor perilaku.

# **2. Import Library**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

print('Libraries imported successfully.')

# **3. Memuat Dataset**

In [ ]:
# Load dataset
df = pd.read_csv('social_media_mental_health.csv')

print('Shape:', df.shape)
print('\nSample data:')
df.head()

In [ ]:
# Cek kolom dan tipe data
print('Columns & dtypes:')
print(df.dtypes)
print('\nBasic info:')
df.info()

# **4. Exploratory Data Analysis (EDA)**

## 4.1 Statistik Deskriptif

In [ ]:
# Statistik deskriptif numerik
df.describe()

In [ ]:
# Statistik deskriptif kolom kategorikal
cat_cols = df.select_dtypes(include='object').columns
print('Categorical columns:', list(cat_cols))
for col in cat_cols:
    print(f'\n{col}:')
    print(df[col].value_counts())

## 4.2 Missing Values & Duplikat

In [ ]:
# Cek missing values
print('Missing values per column:')
print(df.isnull().sum())
print(f'\nTotal missing: {df.isnull().sum().sum()}')

# Cek duplikat
print(f'\nDuplicate rows: {df.duplicated().sum()}')

## 4.3 Distribusi Target (PHQ_9_Severity)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribusi PHQ_9_Severity
order = ['None', 'Minimal', 'Mild', 'Moderate', 'Moderately Severe', 'Severe']
existing_order = [x for x in order if x in df['PHQ_9_Severity'].unique()]
df['PHQ_9_Severity'].value_counts().reindex(existing_order).plot(
    kind='bar', ax=axes[0], color='steelblue', edgecolor='black'
)
axes[0].set_title('Distribusi PHQ_9_Severity (Target)', fontsize=13)
axes[0].set_xlabel('Severity')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)

# Distribusi PHQ_9_Score
axes[1].hist(df['PHQ_9_Score'], bins=28, color='coral', edgecolor='black')
axes[1].set_title('Distribusi PHQ_9_Score', fontsize=13)
axes[1].set_xlabel('PHQ-9 Score')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

## 4.4 Distribusi Fitur Numerik

In [ ]:
num_cols = ['Age', 'Daily_Screen_Time_Hours', 'Sleep_Duration_Hours', 'GAD_7_Score', 'PHQ_9_Score']
df[num_cols].hist(bins=20, figsize=(16, 8), color='teal', edgecolor='black')
plt.suptitle('Distribusi Fitur Numerik', fontsize=14)
plt.tight_layout()
plt.show()

## 4.5 Korelasi Antar Fitur Numerik

In [ ]:
plt.figure(figsize=(10, 6))
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', square=True, linewidths=0.5)
plt.title('Heatmap Korelasi Fitur Numerik', fontsize=13)
plt.tight_layout()
plt.show()

## 4.6 Analisis Gender vs PHQ-9

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot PHQ_9_Score per Gender
df.boxplot(column='PHQ_9_Score', by='Gender', ax=axes[0])
axes[0].set_title('PHQ-9 Score by Gender')
axes[0].set_xlabel('Gender')
axes[0].set_ylabel('PHQ-9 Score')

# Boxplot GAD_7_Score per Gender
df.boxplot(column='GAD_7_Score', by='Gender', ax=axes[1])
axes[1].set_title('GAD-7 Score by Gender')
axes[1].set_xlabel('Gender')
axes[1].set_ylabel('GAD-7 Score')

plt.tight_layout()
plt.show()

## 4.7 Screen Time vs PHQ-9 Score

In [ ]:
plt.figure(figsize=(10, 5))
sns.scatterplot(
    data=df, x='Daily_Screen_Time_Hours', y='PHQ_9_Score',
    hue='Late_Night_Usage', alpha=0.5, palette='Set1'
)
plt.title('Daily Screen Time vs PHQ-9 Score (colored by Late Night Usage)', fontsize=12)
plt.xlabel('Daily Screen Time (Hours)')
plt.ylabel('PHQ-9 Score')
plt.tight_layout()
plt.show()

## 4.8 Platform vs PHQ-9 Severity

In [ ]:
plt.figure(figsize=(12, 5))
platform_phq = df.groupby('Primary_Platform')['PHQ_9_Score'].mean().sort_values(ascending=False)
platform_phq.plot(kind='bar', color='mediumpurple', edgecolor='black')
plt.title('Rata-rata PHQ-9 Score per Platform', fontsize=12)
plt.xlabel('Platform')
plt.ylabel('Mean PHQ-9 Score')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

# **5. Data Preprocessing**

## 5.1 Hapus Kolom Tidak Relevan

In [ ]:
# Hapus User_ID (identifier, tidak informatif untuk model)
# Hapus PHQ_9_Score & GAD_7_Score karena sudah direpresentasikan oleh severity (menghindari data leakage)
df_clean = df.drop(columns=['User_ID', 'PHQ_9_Score', 'GAD_7_Score'])

print('Shape setelah drop kolom:', df_clean.shape)
print('Kolom tersisa:', list(df_clean.columns))

## 5.2 Tangani Missing Values

In [ ]:
# Cek ulang missing values
print('Missing values:')
print(df_clean.isnull().sum())

# Isi missing values numerik dengan median, kategorikal dengan modus
for col in df_clean.select_dtypes(include=np.number).columns:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col].fillna(df_clean[col].median(), inplace=True)

for col in df_clean.select_dtypes(include='object').columns:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col].fillna(df_clean[col].mode()[0], inplace=True)

print('\nMissing values after handling:', df_clean.isnull().sum().sum())

## 5.3 Hapus Duplikat

In [ ]:
before = len(df_clean)
df_clean = df_clean.drop_duplicates()
after = len(df_clean)
print(f'Rows removed (duplicates): {before - after}')
print(f'Shape setelah deduplikasi: {df_clean.shape}')

## 5.4 Encoding Kolom Kategorikal

In [ ]:
# Label Encoding untuk kolom kategorikal (termasuk target)
le = LabelEncoder()
cat_cols_to_encode = df_clean.select_dtypes(include='object').columns.tolist()
print('Kolom yang di-encode:', cat_cols_to_encode)

df_encoded = df_clean.copy()
label_mappings = {}

for col in cat_cols_to_encode:
    df_encoded[col] = le.fit_transform(df_encoded[col])
    label_mappings[col] = dict(zip(le.classes_, le.transform(le.classes_)))

print('\nMapping PHQ_9_Severity:')
print(label_mappings.get('PHQ_9_Severity', 'N/A'))

df_encoded.head()

## 5.5 Deteksi & Tangani Outlier (IQR)

In [ ]:
# Visualisasi outlier sebelum penanganan
num_features = ['Age', 'Daily_Screen_Time_Hours', 'Sleep_Duration_Hours']
fig, axes = plt.subplots(1, len(num_features), figsize=(14, 4))
for i, col in enumerate(num_features):
    axes[i].boxplot(df_encoded[col])
    axes[i].set_title(col)
plt.suptitle('Boxplot Sebelum Penanganan Outlier')
plt.tight_layout()
plt.show()

# Clipping outlier dengan batas IQR
df_no_outlier = df_encoded.copy()
for col in num_features:
    Q1 = df_no_outlier[col].quantile(0.25)
    Q3 = df_no_outlier[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df_no_outlier[col] = df_no_outlier[col].clip(lower, upper)

print('Outlier ditangani dengan clipping IQR.')
print('Shape:', df_no_outlier.shape)

## 5.6 Pisahkan Fitur & Target, Lalu Normalisasi

In [ ]:
# Pisahkan X dan y
X = df_no_outlier.drop(columns=['PHQ_9_Severity'])
y = df_no_outlier['PHQ_9_Severity']

print('X shape:', X.shape)
print('y shape:', y.shape)
print('Target distribution:')
print(y.value_counts())

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train size: {X_train.shape}, Test size: {X_test.shape}')

# Normalisasi fitur numerik
scaler = StandardScaler()
num_cols_to_scale = ['Age', 'Daily_Screen_Time_Hours', 'Sleep_Duration_Hours']
X_train[num_cols_to_scale] = scaler.fit_transform(X_train[num_cols_to_scale])
X_test[num_cols_to_scale] = scaler.transform(X_test[num_cols_to_scale])

print('Normalisasi selesai.')
X_train.head()

## 5.7 Simpan Dataset Preprocessing

In [ ]:
import os
os.makedirs('social_media_mental_health_preprocessing', exist_ok=True)

# Gabungkan kembali train dan test untuk disimpan
df_train = pd.concat([X_train, y_train], axis=1)
df_test  = pd.concat([X_test,  y_test],  axis=1)

df_train.to_csv('social_media_mental_health_preprocessing/train.csv', index=False)
df_test.to_csv( 'social_media_mental_health_preprocessing/test.csv',  index=False)

print('Dataset preprocessing disimpan:')
print('  social_media_mental_health_preprocessing/train.csv')
print('  social_media_mental_health_preprocessing/test.csv')

## 5.8 Verifikasi Data Siap Latih

In [ ]:
print('=== Verifikasi Data Preprocessing ===')
print(f'X_train: {X_train.shape}')
print(f'X_test : {X_test.shape}')
print(f'Missing values di X_train: {X_train.isnull().sum().sum()}')
print(f'Missing values di X_test : {X_test.isnull().sum().sum()}')
print('\nSample X_train:')
print(X_train.head(3))
print('\nDistribusi y_train:')
print(y_train.value_counts())